In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
!pip install datasets transformers evaluate fvcore

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 2.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 11.7 MB/s eta 0:00:00
  Created wheel for fvcore: filename=fvcore-0.1.5.post20221221-py3-none-any.whl size=61397 sha256=3d3e5aa073b59ab8d01171d4031d668863b9c293132de3b60d45b236fa89d3e5
  Stored in directory: /root/.cache/pip/wheels/65/71/95/3b8fde5c65c6e4a806e0867c1651dcc71a1cb2f3430e8f355f
  Created wheel for iopath: filename=iopath-0.1.10-py3-none-any.whl size=31527 sha256=78812c1abf46529f9292bc72fc7a9cfa4daa4e8427b803d37ad53c0470ab519a
  Stored in directory: /root/.cache/pip/wheels/ba/5e/16/6117f8fe7e9c0c161a795e10d94645ebcf301ccbd01f66d8ec
Successfully built fvcore iopath
  Attempting uninstall: fsspec
    Fou

In [2]:
#data download and preprocessing

from datasets import load_dataset
from datasets import concatenate_datasets
from transformers import AutoTokenizer
from pympler import asizeof
from torch.utils.data import DataLoader
from collections import Counter

# Load datasets
mnli_dataset = load_dataset("glue", "mnli")
snli_dataset = load_dataset("snli")

# Remove 'idx' column from MNLI
mnli_dataset = {
    split: ds.remove_columns("idx") 
    for split, ds in mnli_dataset.items()
}

# Function to filter out invalid labels
def filter_valid(example):
    return example["label"] != -1

# Filter all SNLI splits - WITH MULTIPROCESSING
snli_dataset = {
    split: ds.filter(filter_valid, num_proc=4)  # Add num_proc
    for split, ds in snli_dataset.items()
}

# Filter all MNLI splits except test split - WITH MULTIPROCESSING
mnli_dataset = {
    split: ds.filter(filter_valid, num_proc=4) if "test" not in split else ds  # Add num_proc
    for split, ds in mnli_dataset.items()
}


train_dataset = concatenate_datasets([
    snli_dataset['train'], 
    mnli_dataset['train']
]).shuffle(seed=42)


validation_dataset = concatenate_datasets([
    snli_dataset['validation'], 
    mnli_dataset['validation_matched']
]).shuffle(seed=42)


validation_mismatched_dataset = mnli_dataset['validation_mismatched']


# Check the dataset structure
print(mnli_dataset)
print(snli_dataset)

# Initialize the tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# tokenize_function with dynamic padding
def tokenize_function(example):
    return tokenizer(
        example["premise"],
        example["hypothesis"],
        padding=False,
        truncation=True,
        max_length=128
    )

# Tokenize each dataset separately - WITH MULTIPROCESSING
tokenized_train = train_dataset.map(tokenize_function, batched=True, num_proc=4)
tokenized_validation = validation_dataset.map(tokenize_function, batched=True, num_proc=4)
tokenized_validation_mismatched = validation_mismatched_dataset.map(tokenize_function, batched=True, num_proc=4)

# Tokenize test sets - WITH MULTIPROCESSING
tokenized_test_mnli_matched = mnli_dataset["test_matched"].map(tokenize_function, batched=True, num_proc=4)
tokenized_test_mnli_mismatched = mnli_dataset["test_mismatched"].map(tokenize_function, batched=True, num_proc=4)
tokenized_test_snli = snli_dataset["test"].map(tokenize_function, batched=True, num_proc=4)


# Change dataset format to use "labels" instead of "label"
tokenized_train = tokenized_train.rename_column("label", "labels")
tokenized_validation = tokenized_validation.rename_column("label", "labels")
tokenized_validation_mismatched = tokenized_validation_mismatched.rename_column("label", "labels")

tokenized_test_mnli_matched = tokenized_test_mnli_matched.rename_column("label", "labels")
tokenized_test_mnli_mismatched = tokenized_test_mnli_mismatched.rename_column("label", "labels")
tokenized_test_snli = tokenized_test_snli.rename_column("label", "labels")

# Set format for PyTorch
tokenized_train.set_format("torch", columns=["input_ids", "token_type_ids", "attention_mask", "labels"])
tokenized_validation.set_format("torch", columns=["input_ids", "token_type_ids", "attention_mask", "labels"])
tokenized_validation_mismatched.set_format("torch", columns=["input_ids", "token_type_ids", "attention_mask", "labels"])

tokenized_test_mnli_matched.set_format("torch", columns=["input_ids", "token_type_ids", "attention_mask", "labels"])
tokenized_test_mnli_mismatched.set_format("torch", columns=["input_ids", "token_type_ids", "attention_mask", "labels"])
tokenized_test_snli.set_format("torch", columns=["input_ids", "token_type_ids", "attention_mask", "labels"])


print(f"Train dataset size: {len(tokenized_train)}")
print(f"Validation dataset size: {len(tokenized_validation)}")
print(f"Validation mismatched size: {len(tokenized_validation_mismatched)}")

# attention_mask check 
print(tokenized_train[0]["attention_mask"])

# Check the train dataset dtype and size
size_in_bytes = asizeof.asizeof(tokenized_train)
size_in_mb = size_in_bytes / (1024 * 1024)
print(f"Total size of tokenized_train: {size_in_mb:.2f} MB")

# Check label distribution
train_labels = tokenized_train["labels"].tolist()
print(f"Label distribution in train: {Counter(train_labels)}")

print(tokenized_train[0]["input_ids"].shape, tokenized_train[0]["input_ids"].dtype)
print(tokenized_train[0]["token_type_ids"].shape, tokenized_train[0]["token_type_ids"].dtype)
print(tokenized_train[0]["attention_mask"].shape, tokenized_train[0]["attention_mask"].dtype)
print(tokenized_train[0]["labels"].shape, tokenized_train[0]["labels"].dtype)

# Check data types of each column
sample = tokenized_train[0]
for key, value in sample.items():
    print(f"Column: {key}, Data Type: {value.dtype}")

README.md: 0.00B [00:00, ?B/s]

mnli/train-00000-of-00001.parquet:   0%|          | 0.00/52.2M [00:00<?, ?B/s]

mnli/validation_matched-00000-of-00001.p(…):   0%|          | 0.00/1.21M [00:00<?, ?B/s]

mnli/validation_mismatched-00000-of-0000(…):   0%|          | 0.00/1.25M [00:00<?, ?B/s]

mnli/test_matched-00000-of-00001.parquet:   0%|          | 0.00/1.22M [00:00<?, ?B/s]

...

Map (num_proc=4):   0%|          | 0/9832 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/9796 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/9847 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/9824 [00:00<?, ? examples/s]

Train dataset size: 942069
Validation dataset size: 19657
Validation mismatched size: 9832
tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1])
Total size of tokenized_train: 1259.59 MB
Label distribution in train: Counter({0: 314315, 2: 314090, 1: 313664})
torch.Size([13]) torch.int64
torch.Size([13]) torch.int64
torch.Size([13]) torch.int64
torch.Size([]) torch.int64
Column: labels, Data Type: torch.int64
Column: input_ids, Data Type: torch.int64
Column: token_type_ids, Data Type: torch.int64
Column: attention_mask, Data Type: torch.int64


In [25]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import time
import wandb
import random
import os
import inspect
from tqdm import tqdm
from contextlib import nullcontext
from dataclasses import dataclass
from torch.utils.data import Dataset, DataLoader
from torch.utils.checkpoint import checkpoint



# ------------------------------------dynamic tanh---------------------------------------------------
class DyT(nn.Module):
    def __init__(self, num_features, alpha_init_value=0.5):
        super().__init__()
        self.alpha = nn.Parameter(torch.ones(1) * alpha_init_value)
        self.weight = nn.Parameter(torch.ones(num_features))
        self.bias = nn.Parameter(torch.zeros(num_features))
    
    def forward(self, x):
        x = torch.tanh(self.alpha * x)
        return x * self.weight + self.bias

# ------------------------------------MHA---------------------------------------------------

class Attention(nn.Module):
    """Multi-head attention"""
    def __init__(self, config):
        super().__init__()
        assert config.n_embed % config.n_head == 0, "Embedding dim must be divisible by num heads"

        self.n_head = config.n_head
        self.n_embed = config.n_embed
        self.head_dim = config.n_embed // config.n_head

        self.qkv_proj = nn.Linear(config.n_embed, 3 * config.n_embed)
        self.output_proj = nn.Linear(config.n_embed, config.n_embed)
        self.dropout = nn.Dropout(config.dropout)



    def forward(self, x: torch.Tensor, attention_mask: torch.Tensor = None) -> torch.Tensor:
        B, T, C = x.shape

        # Project to queries, keys, values
        qkv = self.qkv_proj(x)
        q, k, v = qkv.split(self.n_embed, dim=2)

        # Reshape for multi-head attention
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)  # (B, nh, T, hs)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)  # (B, nh, T, hs)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)  # (B, nh, T, hs)


        # Process attention mask
        attn_mask = None
        if attention_mask is not None:
            # Expand from (B, T) to (B, 1, 1, T) for broadcasting
            attn_mask = attention_mask.unsqueeze(1).unsqueeze(2)

        # Compute attention
        y = F.scaled_dot_product_attention(
            q, k, v,
            attn_mask=attn_mask,
            is_causal=False
        )

        # Reshape and project
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.dropout(self.output_proj(y))

        return y

# -----------------------------------Expert--------------------------------------------------

class Expert(nn.Module):
    def __init__(self,config):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(config.n_embed,2*config.n_embed),
            nn.GELU(),
            nn.Linear(2*config.n_embed,config.n_embed),
            nn.Dropout(config.dropout)
        )

    def forward(self,x):
        return self.net(x)

# ------------------------------------MLP--------------------------------------------------

class MLP(nn.Module):
    def __init__(self,config):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(config.n_embed,4*config.n_embed),
            nn.GELU(),
            nn.Linear(4*config.n_embed,config.n_embed),
            nn.Dropout(config.dropout)
        )

    def forward(self,x):
        return self.net(x)

# ------------------------------------NoisyTopkRouter--------------------------------------------------

class NoisyTopkRouter(nn.Module):
    def __init__(self, config):
        super(NoisyTopkRouter, self).__init__()
        self.top_k = config.top_k
        # Layer for router logits
        self.topkroute_linear = nn.Linear(config.n_embed, config.num_experts)
        self.noise_linear = nn.Linear(config.n_embed, config.num_experts)
    
    def forward(self, mh_output):
        # mh_ouput is the output tensor from multihead self attention block
        logits = self.topkroute_linear(mh_output)
        # Noise logits
        noise_logits = self.noise_linear(mh_output)
        # Adding scaled unit gaussian noise to the logits
        noise = torch.randn_like(logits) * F.softplus(noise_logits)
        noisy_logits = logits + noise
        top_k_logits, indices = noisy_logits.topk(self.top_k, dim=-1)
        zeros = torch.full_like(noisy_logits, float('-inf'))
        sparse_logits = zeros.scatter(-1, indices, top_k_logits)
        router_output = F.softmax(sparse_logits, dim=-1)
        return router_output, indices, logits


# ------------------------------------SparseMoE--------------------------------------------------

class SparseMoE(nn.Module):
    def __init__(self, config):
        super(SparseMoE, self).__init__()
        self.router = NoisyTopkRouter(config)
        self.experts = nn.ModuleList([Expert(config) for _ in range(config.num_experts)])
        self.top_k = config.top_k
        self.capacity_factor = config.capacity_factor
        self.num_experts = config.num_experts
        
        # Load balancing parameters
        self.use_load_balancing = getattr(config, "use_load_balancing", True)
        self.load_balance_weight = getattr(config, "load_balance_weight", 0.01)
        self.aux_loss = 0.0
    
    def _compute_load_balancing_loss(self, router_logits, expert_indices):
        """
        Computes auxiliary load balancing loss to encourage uniform expert utilization
        
        Args:
            router_logits: Original router logits before noise [batch_size, seq_len, num_experts]
            expert_indices: Expert assignments [batch_size, seq_len, top_k]
        """
        # Calculate fraction of tokens assigned to each expert
        batch_size, seq_len, _ = router_logits.shape
        
        # More efficient one-hot representation of expert assignment
        # Shape: [batch_size, seq_len, top_k, num_experts]
        one_hot = F.one_hot(expert_indices, num_classes=self.num_experts)
        # Sum across top-k dimension to get mask of shape [batch_size, seq_len, num_experts]
        mask = one_hot.sum(dim=2).float()
        
        # Calculate the fraction of tokens routed to each expert
        routing_probs = mask.mean(dim=[0, 1])  # Shape [num_experts]
        
        # Calculate router probabilities from logits
        router_probs = F.softmax(router_logits, dim=-1)  # [batch_size, seq_len, num_experts]
        
        # Calculate mean probability assigned to each expert
        router_probs = router_probs.mean(dim=[0, 1])  # Shape [num_experts]
        
        # Loss is the dot product between router probability and expert assignment fraction
        # This encourages the router to distribute tokens uniformly
        loss = routing_probs @ router_probs * self.num_experts
        
        return loss
    
    def forward(self, x):
        """
        Forward pass with load balancing
        
        Args:
            x: Input tensor of shape [batch_size, seq_len, n_embed]
        """
        batch_size, seq_len, embed_dim = x.shape
        total_tokens = batch_size * seq_len
        
        # Get routing probabilities and expert assignments
        router_probs, indices, router_logits = self.router(x)
        
        if self.training and self.use_load_balancing:
            self.aux_loss = self._compute_load_balancing_loss(router_logits, indices)
        else:
            self.aux_loss = 0.0
        
        # Reshape input for expert processing
        # [batch_size, seq_len, embed_dim] -> [batch_size * seq_len, embed_dim]
        flat_x = x.reshape(-1, embed_dim)
        
        # Calculate expert capacity - how many tokens each expert should process
        # We use capacity_factor > 1.0 to allow for some imbalance
        tokens_per_expert = int((total_tokens * self.top_k / self.num_experts) * self.capacity_factor)
        
        # Initialize output tensor
        combined_output = torch.zeros_like(flat_x)
        
        # Create a mask for combining outputs, shape [batch_size * seq_len, num_experts]
        combine_weights = router_probs.view(-1, self.num_experts)
        
        # Process each expert
        for expert_idx, expert in enumerate(self.experts):
            # Find which tokens go to this expert
            # Shape: [batch_size, seq_len]
            expert_mask = (indices == expert_idx).any(dim=-1)
            # Flatten to [batch_size * seq_len]
            flat_mask = expert_mask.reshape(-1)
            # Get indices of tokens assigned to this expert
            token_indices = flat_mask.nonzero(as_tuple=True)[0]
            
            # Only process if we have tokens for this expert
            if token_indices.numel() > 0:
                # Apply capacity constraints if needed
                if token_indices.numel() > tokens_per_expert:
                    # Too many tokens assigned, need to drop some
                    # Get the routing probabilities for this expert
                    expert_probs = combine_weights[token_indices, expert_idx]
                    # Sort by probability
                    sorted_indices = torch.argsort(expert_probs, descending=True)
                    # Keep only tokens within capacity
                    selected_indices = sorted_indices[:tokens_per_expert]
                    
                    # Get the actual indices in the flattened input
                    token_indices = token_indices[selected_indices]
                
                # Process expert computation for selected tokens
                expert_inputs = flat_x[token_indices]
                expert_outputs = expert(expert_inputs)
                expert_weights = combine_weights[token_indices, expert_idx].unsqueeze(-1)
                weighted_outputs = expert_outputs * expert_weights
                combined_output.index_add_(0, token_indices, weighted_outputs)
        
        # Reshape back to original dimensions
        output = combined_output.reshape(batch_size, seq_len, embed_dim)
        
        return output
    
    def get_load_balancing_loss(self):
        """Return the load balancing auxiliary loss"""
        return self.aux_loss * self.load_balance_weight if self.use_load_balancing else 0.0


# ------------------------------------Block---------------------------------------------------

class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.use_moe = config.use_moe
        self.ln1 = DyT(config.n_embed)
        self.attn = Attention(config)
        self.ln2 = DyT(config.n_embed)
        if self.use_moe:
            self.moe = SparseMoE(config)
        else:
            self.moe = MLP(config)

    def forward(self, x, attention_mask=None):
        x = x + self.attn(self.ln1(x), attention_mask)
        x = x + self.moe(self.ln2(x))
        return x


#-----------------------------BERT Embedding---------------------------------------------------------

class BERTEmbedding(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.token_embed = nn.Embedding(config.vocab_size, config.n_embed)
        self.position_embed = nn.Embedding(config.block_size, config.n_embed)
        self.segment_embed = nn.Embedding(2, config.n_embed)  # Only 2 segment types for BERT
        self.ln = nn.LayerNorm(config.n_embed)
        self.dropout = nn.Dropout(config.dropout)
        
    def forward(self, input_ids, segment_ids):
        # Debug: Print shapes to understand the issue
        # print(f"Debug - input_ids shape: {input_ids.shape}")
        # print(f"Debug - segment_ids shape: {segment_ids.shape}")
        
        # seq_len = input_ids.size(1) #B,T
        batch_size, seq_len = input_ids.shape
        pos_ids = torch.arange(seq_len, dtype=torch.long, device=input_ids.device).unsqueeze(0)
        
        token_emb = self.token_embed(input_ids)
        pos_emb = self.position_embed(pos_ids)
        seg_emb = self.segment_embed(segment_ids)
        
        embeddings = token_emb + seg_emb + pos_emb
        return self.dropout(self.ln(embeddings))


class BERT(nn.Module):
    def __init__(self, config):
        super().__init__()
    
        self.use_moe=config.use_moe
        
        self.embedding = BERTEmbedding(config)
        # Cannot use nn.Sequential when we need to pass additional parameters like attention_mask
        self.blocks = nn.ModuleList([Block(config) for _ in range(config.n_layer)])
        self.ln = nn.RMSNorm(config.n_embed)
        self.head = nn.Linear(config.n_embed, config.num_labels)

        # Collect all MoE layers from blocks
        if self.use_moe:
            self.moe_layers = nn.ModuleList([block.moe for block in self.blocks])
        
        # Apply weight initialization
        self.apply(self._init_weights)
        
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

            
    def forward(self, input_ids, segment_ids, targets=None, attention_mask=None):
        x = self.embedding(input_ids, segment_ids)
        
        # Process through each block manually to pass attention_mask
        for block in self.blocks:
            x = block(x, attention_mask)
            
        x = self.ln(x[:, 0, :])  # Take [CLS] token representation for classification
        logits = self.head(x)
        
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits, targets)

            if self.use_moe:
                # Add auxiliary load balancing loss from all MoE layers
                aux_loss = 0.0
                for moe_layer in self.moe_layers:
                    aux_loss += moe_layer.get_load_balancing_loss()
                
                # Combine losses
                loss = loss + aux_loss
        
        return logits, loss
        
        
    def configure_optimizers(self, weight_decay, learning_rate, device, verbose=False):

        # Step 1: Collect trainable parameters (excluding frozen ones)
        param_dict = {pn: p for pn, p in self.named_parameters()}
        param_dict = {pn: p for pn, p in param_dict.items() if p.requires_grad}
    
        # Step 2: Separate weight decay (2D+ tensors) and no decay (biases, LayerNorm)
        decay_params = [p for n, p in param_dict.items() if p.dim() >= 2]  # Weight matrices
        nodecay_params = [p for n, p in param_dict.items() if p.dim() < 2]  # Biases, LayerNorm
    
        # Step 3: Define optimizer parameter groups
        optim_groups = [
            {'params': decay_params, 'weight_decay': weight_decay},  # Apply weight decay
            {'params': nodecay_params, 'weight_decay': 0.0}  # No weight decay
        ]

        fused_available = 'fused' in inspect.signature(torch.optim.AdamW).parameters
        use_fused = fused_available and 'cuda' in device
        
        # Debugging - Print parameter stats (optional)
        if verbose:
            num_decay_params = sum(p.numel() for p in decay_params)
            num_nodecay_params = sum(p.numel() for p in nodecay_params)
            print(f"num decayed parameter tensors: {len(decay_params)}, with {num_decay_params/1e6}M parameters")
            print(f"num non-decayed parameter tensors: {len(nodecay_params)}, with {num_nodecay_params/1e6}M parameters")
            print(f"Using fused AdamW: {use_fused}")

        # Create AdamW optimizer
        optimizer = torch.optim.AdamW(
            optim_groups, 
            lr=learning_rate, 
            betas = (0.9, 0.999),  # Momentum values,  # Momentum values
            eps=1e-8,  # Small epsilon for numerical stability
            fused=use_fused  # Enable fused version if available
        )
    
        return optimizer


random.seed(1337)
torch.manual_seed(1337)
if torch.cuda.is_available():
    torch.cuda.manual_seed(1337)

device = "cuda" if torch.cuda.is_available() else "cpu"


#TF16
torch.set_float32_matmul_precision('high')

In [26]:
import numpy as np
from fvcore.nn import FlopCountAnalysis, flop_count_table

# Prepare real inputs using your tokenizer
text1 = "The man is walking down the street."
text2 = "A person is outside."

# Tokenize (assuming you have your tokenizer loaded)
inputs = tokenizer(
    text1, 
    text2,
    max_length=128,
    padding='max_length',
    truncation=True,
    return_tensors='pt'
)



print(torch.cuda.get_device_name(0))
print(torch.cuda.get_device_capability(0))
print(torch.cuda.is_bf16_supported())


def benchmark_model(model, inputs, config, use_mixed_precision=True):
    
    device = config.device
    seq_len = config.block_size
    
    # Setup mixed precision
    if use_mixed_precision and device == "cuda":
        dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
        print(f"Using mixed precision: {dtype}")
    else:
        dtype = torch.float32
        print("Using full precision: float32")
    
    # # Create sample inputs
    # input_ids = torch.randint(0, config.vocab_size, (1, seq_len), device=device)
    # segment_ids = torch.randint(0, 2, (1, seq_len), device=device)
    # attention_mask = torch.ones(1, seq_len, device=device)

    input_ids = inputs['input_ids'].to(device)
    segment_ids = inputs['token_type_ids'].to(device)
    attention_mask = inputs['attention_mask'].to(device).bool()
    
    model.eval()
    
    print("="*70)
    print("MODEL STATISTICS")
    print("="*70)
    
    # Parameter Count
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total parameters: {total_params:,} ({total_params/1e6:.2f}M)")
    print(f"Trainable parameters: {trainable_params:,} ({trainable_params/1e6:.2f}M)")
    
    # FLOPs Calculation
    print("\n" + "="*70)
    print("FLOPS ANALYSIS")
    print("="*70)
    
    with torch.no_grad():
        flops = FlopCountAnalysis(model, (input_ids, segment_ids, None, attention_mask))
        total_flops = flops.total()
        
        print(f"Total FLOPs per forward pass: {total_flops/1e9:.2f} GFLOPs")
        print(f"FLOPs per sample: {total_flops/1e9:.2f} GFLOPs")
        print(f"Total FLOPs for sequence: {total_flops/1e12:.4f} TFLOPs")
    
    # Memory Usage
    print("\n" + "="*70)
    print("MEMORY USAGE")
    print("="*70)
    
    torch.cuda.reset_peak_memory_stats()
    with torch.no_grad():
        if use_mixed_precision and device == "cuda":
            with torch.autocast(device_type='cuda', dtype=dtype):
                _ = model(input_ids, segment_ids, None, attention_mask)
        else:
            _ = model(input_ids, segment_ids, None, attention_mask)
    
    memory_allocated = torch.cuda.max_memory_allocated() / 1e6
    print(f"Peak GPU memory (batch_size=1): {memory_allocated:.2f} MB")
    
    # Latency Measurement
    print("\n" + "="*70)
    print("LATENCY MEASUREMENT (batch_size=1)")
    print("="*70)
    
    # Warmup
    print("Warming up...")
    with torch.no_grad():
        for _ in range(20):
            if use_mixed_precision and device == "cuda":
                with torch.autocast(device_type='cuda', dtype=dtype):
                    _ = model(input_ids, segment_ids, None, attention_mask)
            else:
                _ = model(input_ids, segment_ids, None, attention_mask)
    torch.cuda.synchronize()
    
    # Actual measurement
    print("Measuring latency...")
    latencies = []
    num_iterations = 100
    
    with torch.no_grad():
        for _ in range(num_iterations):
            torch.cuda.synchronize()
            start = time.perf_counter()
            
            if use_mixed_precision and device == "cuda":
                with torch.autocast(device_type='cuda', dtype=dtype):
                    _ = model(input_ids, segment_ids, None, attention_mask)
            else:
                _ = model(input_ids, segment_ids, None, attention_mask)
            
            torch.cuda.synchronize()
            latencies.append((time.perf_counter() - start) * 1000)  # ms
    
    latencies = np.array(latencies)
    mean_latency = np.mean(latencies)
    std_latency = np.std(latencies)
    print(f"Latency: {mean_latency:.2f} ± {std_latency:.2f} ms")
    
    # Throughput Measurement
    print("\n" + "="*70)
    print("THROUGHPUT MEASUREMENT")
    print("="*70)
    
    batch_sizes = [1, 8, 16, 32]
    throughput_results = []
    
    for batch_size in batch_sizes:
        try:
            input_ids_batch = input_ids.repeat(batch_size, 1)
            segment_ids_batch = segment_ids.repeat(batch_size, 1)
            attention_mask_batch = attention_mask.repeat(batch_size, 1)
            
            # Warmup
            with torch.no_grad():
                for _ in range(10):
                    if use_mixed_precision and device == "cuda":
                        with torch.autocast(device_type='cuda', dtype=dtype):
                            _ = model(input_ids_batch, segment_ids_batch, None, attention_mask_batch)
                    else:
                        _ = model(input_ids_batch, segment_ids_batch, None, attention_mask_batch)
            torch.cuda.synchronize()
            
            # Measure
            num_iterations = 50
            torch.cuda.synchronize()
            start = time.perf_counter()
            
            with torch.no_grad():
                for _ in range(num_iterations):
                    if use_mixed_precision and device == "cuda":
                        with torch.autocast(device_type='cuda', dtype=dtype):
                            _ = model(input_ids_batch, segment_ids_batch, None, attention_mask_batch)
                    else:
                        _ = model(input_ids_batch, segment_ids_batch, None, attention_mask_batch)
            
            torch.cuda.synchronize()
            elapsed_time = time.perf_counter() - start
            
            samples_per_sec = (batch_size * num_iterations) / elapsed_time
            tokens_per_sec = samples_per_sec * seq_len
            
            # Memory check
            torch.cuda.reset_peak_memory_stats()
            with torch.no_grad():
                if use_mixed_precision and device == "cuda":
                    with torch.autocast(device_type='cuda', dtype=dtype):
                        _ = model(input_ids_batch, segment_ids_batch, None, attention_mask_batch)
                else:
                    _ = model(input_ids_batch, segment_ids_batch, None, attention_mask_batch)
            memory_used = torch.cuda.max_memory_allocated() / 1e9
            
            throughput_results.append({
                'batch_size': batch_size,
                'samples_per_sec': samples_per_sec,
                'tokens_per_sec': tokens_per_sec,
                'memory_gb': memory_used
            })
            
            print(f"Batch size {batch_size:3d}: {samples_per_sec:7.2f} samples/sec | "
                  f"{tokens_per_sec:10.2f} tokens/sec | Memory: {memory_used:.2f} GB")
            
        except RuntimeError as e:
            if "out of memory" in str(e):
                print(f"Batch size {batch_size:3d}: OOM (Out of Memory)")
                torch.cuda.empty_cache()
                break
            else:
                raise e
    
    # Performance Summary
    print("\n" + "="*70)
    print("PERFORMANCE SUMMARY")
    print("="*70)
    
    if throughput_results:
        max_throughput = max(throughput_results, key=lambda x: x['tokens_per_sec'])
        print(f"Peak throughput: {max_throughput['tokens_per_sec']:.2f} tokens/sec "
              f"(batch_size={max_throughput['batch_size']})")
        print(f"Single sample latency: {mean_latency:.2f} ± {std_latency:.2f} ms")
        print(f"FLOPs per forward pass: {total_flops/1e9:.2f} GFLOPs")
        
        # Theoretical achieved FLOPS
        mean_latency_sec = mean_latency / 1000
        theoretical_flops = total_flops / mean_latency_sec / 1e12
        print(f"Achieved compute: {theoretical_flops:.2f} TFLOPS")
    
    return throughput_results


Tesla P100-PCIE-16GB
(6, 0)
True


In [33]:
@dataclass
class BERTConfig23:
    block_size: int = 128
    vocab_size: int = 30522
    n_layer: int = 8
    n_head: int = 4
    n_embed: int = 256
    dropout: float = 0.1
    num_labels: int = 3
    use_full_precision: bool = False
    use_moe: bool = True
    top_k: int = 2
    capacity_factor: float = 1.15
    num_experts: int = 6
    use_load_balancing: bool = True
    load_balance_weight: float = 0.01
    device: str = "cuda" if torch.cuda.is_available() else "cpu"

config=BERTConfig23()
model=BERT(config).to(device)
results_mixed = benchmark_model(model, inputs, config, use_mixed_precision=True)

Using mixed precision: torch.bfloat16
MODEL STATISTICS
Total parameters: 22,606,451 (22.61M)
Trainable parameters: 22,606,451 (22.61M)

FLOPS ANALYSIS
Total FLOPs per forward pass: 0.80 GFLOPs
FLOPs per sample: 0.80 GFLOPs
Total FLOPs for sequence: 0.0008 TFLOPs

MEMORY USAGE
Peak GPU memory (batch_size=1): 220.91 MB

LATENCY MEASUREMENT (batch_size=1)
Warming up...
Measuring latency...
Latency: 28.42 ± 0.56 ms

THROUGHPUT MEASUREMENT
Batch size   1:   35.41 samples/sec |    4532.50 tokens/sec | Memory: 0.22 GB
Batch size   8:  273.77 samples/sec |   35042.72 tokens/sec | Memory: 0.23 GB
Batch size  16:  548.77 samples/sec |   70242.48 tokens/sec | Memory: 0.24 GB
Batch size  32:  896.92 samples/sec |  114805.23 tokens/sec | Memory: 0.27 GB

PERFORMANCE SUMMARY
Peak throughput: 114805.23 tokens/sec (batch_size=32)
Single sample latency: 28.42 ± 0.56 ms
FLOPs per forward pass: 0.80 GFLOPs
Achieved compute: 0.03 TFLOPS


In [34]:
@dataclass
class BERTConfig45:
    block_size:int = 128
    # batch_size:int = 32 
    vocab_size:int = 30522
    n_layer:int = 8 #8 12 16
    n_head:int = 6 #8,12,16
    n_embed:int = 384 # 256 512 576 768,1024
    dropout:float = 0.1
    num_labels:int = 3
    use_full_precision:bool=False
    # mixture of experts
    use_moe:bool=True
    top_k:int = 2
    capacity_factor:float = 1.15
    num_experts:int = 6
    use_load_balancing: bool = True
    load_balance_weight: float = 0.01
    device:str = "cuda" if torch.cuda.is_available() else "cpu"


config=BERTConfig45()
model=BERT(config).to(device)
results_mixed = benchmark_model(model, inputs, config, use_mixed_precision=True)

Using mixed precision: torch.bfloat16
MODEL STATISTICS
Total parameters: 44,919,667 (44.92M)
Trainable parameters: 44,919,667 (44.92M)

FLOPS ANALYSIS
Total FLOPs per forward pass: 1.81 GFLOPs
FLOPs per sample: 1.81 GFLOPs
Total FLOPs for sequence: 0.0018 TFLOPs

MEMORY USAGE
Peak GPU memory (batch_size=1): 350.19 MB

LATENCY MEASUREMENT (batch_size=1)
Warming up...
Measuring latency...
Latency: 29.51 ± 0.53 ms

THROUGHPUT MEASUREMENT
Batch size   1:   33.39 samples/sec |    4273.78 tokens/sec | Memory: 0.35 GB
Batch size   8:  262.53 samples/sec |   33603.57 tokens/sec | Memory: 0.36 GB
Batch size  16:  439.58 samples/sec |   56266.37 tokens/sec | Memory: 0.38 GB
Batch size  32:  610.09 samples/sec |   78091.76 tokens/sec | Memory: 0.42 GB

PERFORMANCE SUMMARY
Peak throughput: 78091.76 tokens/sec (batch_size=32)
Single sample latency: 29.51 ± 0.53 ms
FLOPs per forward pass: 1.81 GFLOPs
Achieved compute: 0.06 TFLOPS


In [35]:
@dataclass
class BERTConfig97:
    block_size:int = 128
    # batch_size:int = 32 
    vocab_size:int = 30522
    n_layer:int = 11 #8 12 16
    n_head:int = 8 #8,12,16
    n_embed:int = 512 # 256 512 576 768,1024
    dropout:float = 0.1
    num_labels:int = 3
    use_full_precision:bool=False
    # mixture of experts
    use_moe:bool=True
    top_k:int = 2
    capacity_factor:float = 1.15
    num_experts:int = 6
    use_load_balancing: bool = True
    load_balance_weight: float = 0.01
    device:str = "cuda" if torch.cuda.is_available() else "cpu"


config=BERTConfig97()
model=BERT(config).to(device)
results_mixed = benchmark_model(model, config, use_mixed_precision=True)

Using mixed precision: torch.bfloat16
MODEL STATISTICS
Total parameters: 96,651,421 (96.65M)
Trainable parameters: 96,651,421 (96.65M)

FLOPS ANALYSIS
Total FLOPs per forward pass: 4.41 GFLOPs
FLOPs per sample: 4.41 GFLOPs
Total FLOPs for sequence: 0.0044 TFLOPs

MEMORY USAGE
Peak GPU memory (batch_size=1): 662.87 MB

LATENCY MEASUREMENT (batch_size=1)
Warming up...
Measuring latency...
Latency: 43.12 ± 0.48 ms

THROUGHPUT MEASUREMENT
Batch size   1:   22.96 samples/sec |    2938.76 tokens/sec | Memory: 0.66 GB
Batch size   8:  167.96 samples/sec |   21498.53 tokens/sec | Memory: 0.68 GB
Batch size  16:  248.41 samples/sec |   31796.60 tokens/sec | Memory: 0.70 GB
Batch size  32:  322.80 samples/sec |   41318.28 tokens/sec | Memory: 0.75 GB

PERFORMANCE SUMMARY
Peak throughput: 41318.28 tokens/sec (batch_size=32)
Single sample latency: 43.12 ± 0.48 ms
FLOPs per forward pass: 4.41 GFLOPs
Achieved compute: 0.10 TFLOPS
